# Vision Transformers & Multimodal Models for Medical Imaging

**From ViT-MAE to Multimodal Understanding & Generation**

This notebook teaches two core ideas in modern medical-image AI:

**Part I — Vision Transformer (ViT) with Masked Autoencoders (MAE)**
- How ViT turns images into patch sequences (like words in a sentence)
- How MAE learns by masking and reconstructing patches — no labels needed
- Fine-tuning a pretrained MAE on chest X-rays

**Part II — Multimodal Foundation Model (Janus-Pro-1B)**
- How a single model can *understand* images and *generate* images
- Image captioning and visual question answering on chest X-rays
- Text-to-image generation of medical images

---

| Section | Topic |
|---------|-------|
| **Part I** | **ViT-MAE on Chest X-Rays** |
| 0 | Setup |
| 1 | Load Chest X-Ray Images |
| 2 | Load Pretrained ViT-MAE & Visualize Architecture |
| 3 | How MAE Masking Works |
| 4 | Reconstructions Before Training |
| 5 | Fine-Tune MAE on Chest X-Rays |
| 6 | Reconstructions After Training |
| 7 | Before vs After Comparison |
| **Part II** | **Multimodal Foundation Model** |
| 8 | Intro to Multimodal Models |
| 9 | Load Janus-Pro-1B & Visualize Mixed-Modal Input |
| 10 | Image Captioning |
| 11 | Visual Question Answering |
| 12 | Text-to-Image Generation |
| 13 | Interleaved Generation |
| 14 | Quick Fine-Tuning on CXR |
| 15 | Summary |

> **Colab:** Use *Runtime → Change runtime type → T4 GPU* for faster training.

---
## Section 0: Setup

In [47]:
# ── Section 0: Setup ─────────────────────────────────────────────────────
import sys, subprocess

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# Core ML & vision
# transformers>=4.47 breaks Janus-Pro (missing all_tied_weights_keys); pin to <4.47
pip_install(["transformers>=4.38,<4.47", "datasets", "accelerate", "torchvision", "huggingface_hub"])
pip_install(["Pillow>=10.0,<11.0"])

# Janus-Pro (multimodal model from DeepSeek)
pip_install(["git+https://github.com/deepseek-ai/Janus.git"])

# PEFT for LoRA fine-tuning
pip_install(["peft"])

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from datasets import load_dataset
import transformers
from transformers import (
    ViTImageProcessor,
    ViTMAEForPreTraining,
    ViTMAEConfig,
    Trainer,
    TrainingArguments,
)

print(f"torch: {torch.__version__}")
print(f"transformers: {transformers.__version__}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

plt.rcParams.update({"figure.figsize": (10, 5), "font.size": 12})
USE_PRECOMPUTED = False

---
## Section 1: Load Chest X-Ray Images

We use a **public chest X-ray dataset** from HuggingFace. Chest X-rays (CXR) are the most
common medical imaging modality — every hospital produces thousands per day.

- **No labels needed** — MAE is self-supervised (learns by reconstructing masked patches)
- We load a small subset for fast iteration on Colab

In [48]:
# Load chest X-ray images from HuggingFace
print("Loading chest X-ray dataset...")
cxr_dataset = load_dataset("itsanmolgupta/mimic-cxr-dataset", split="train")

# Identify image and text columns
image_col = "image" if "image" in cxr_dataset.column_names else cxr_dataset.column_names[0]
text_col = None
for col in ["text", "caption", "report", "findings", "impression"]:
    if col in cxr_dataset.column_names:
        text_col = col
        break
if text_col is None:
    for col in cxr_dataset.column_names:
        if col != image_col:
            text_col = col
            break

print(f"Columns: {cxr_dataset.column_names}")
print(f"Image column: {image_col}, Text column: {text_col}")
print(f"Total examples: {len(cxr_dataset)}")

# Take a manageable subset and split 80/20
subset = cxr_dataset.shuffle(seed=SEED).select(range(min(600, len(cxr_dataset))))
N_TRAIN = int(len(subset) * 0.8)
train_ds = subset.select(range(N_TRAIN))
val_ds = subset.select(range(N_TRAIN, len(subset)))

print(f"Using {len(train_ds)} train, {len(val_ds)} val")

In [49]:
# Helper: ensure images are RGB (some CXR are grayscale)
def to_rgb(img):
    if not isinstance(img, Image.Image):
        img = Image.open(img)
    return img.convert("RGB")

# Gallery of chest X-ray samples
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flatten()):
    if i < len(train_ds):
        img = to_rgb(train_ds[i][image_col])
        ax.imshow(img, cmap="gray")
    ax.axis("off")
plt.suptitle("Sample Chest X-Rays", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Section 2: Understanding the Vision Transformer (ViT)

### How ViT Works — Images as Sequences

A **Vision Transformer (ViT)** processes images completely differently from CNNs:

| Step | What happens | Analogy to NLP |
|------|-------------|----------------|
| 1. **Patch** | Split the image into a grid of small squares (e.g., 16×16 pixels) | Splitting a sentence into words |
| 2. **Flatten & Project** | Each patch is flattened to a vector and linearly projected to an embedding | Word → embedding lookup |
| 3. **Add Position** | Add a learnable positional embedding to each patch embedding | Positional encoding in Transformers |
| 4. **Transformer** | Pass the sequence of patch embeddings through standard Transformer layers | Same self-attention as in GPT/BERT |
| 5. **Output** | Use the output for classification, reconstruction, etc. | Task head on top of Transformer |

```
Image (224×224)
    │
    ▼
┌─────────────────┐
│ Split into 14×14 │  ← 196 patches, each 16×16 pixels
│   grid of patches │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│ Flatten each to  │  ← 16×16×3 = 768-dim vector per patch
│ vector + project │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│ + Position       │  ← So the model knows spatial arrangement
│   Embeddings     │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│ Transformer      │  ← 12 layers of self-attention
│ Encoder Blocks   │
└────────┬────────┘
         │
         ▼
    Output tokens
```

**Key insight:** ViT has **no convolutions** — it relies entirely on self-attention to learn spatial relationships between patches.

In [50]:
# Load pretrained ViT-MAE from HuggingFace
MODEL_ID = "facebook/vit-mae-base"
processor = ViTImageProcessor.from_pretrained(MODEL_ID)
model = ViTMAEForPreTraining.from_pretrained(MODEL_ID).to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
n_enc = sum(p.numel() for p in model.vit.parameters())
n_dec = sum(p.numel() for p in model.decoder.parameters())

print(f"Model: {MODEL_ID}")
print(f"Total parameters:   {n_params:>12,}")
print(f"  Encoder (ViT):    {n_enc:>12,}")
print(f"  Decoder (MAE):    {n_dec:>12,}")
print(f"Image size:         {model.config.image_size}")
print(f"Patch size:         {model.config.patch_size}")
print(f"Number of patches:  {(model.config.image_size // model.config.patch_size) ** 2}")
print(f"Hidden dimension:   {model.config.hidden_size}")
print(f"Encoder layers:     {model.config.num_hidden_layers}")
print(f"Attention heads:    {model.config.num_attention_heads}")
print(f"Mask ratio:         {model.config.mask_ratio}")

### Model Architecture — Encoder and Decoder

The MAE model has two parts:
- **Encoder** (ViT): Processes only the *visible* patches through 12 Transformer layers
- **Decoder**: A smaller Transformer that takes encoder output + mask tokens and reconstructs all patches

Let's print the architecture to see the building blocks:

In [51]:
# Print the model architecture (simplified view)
print("=" * 60)
print("ENCODER (ViT)")
print("=" * 60)
print(f"  Patch Embeddings:")
print(f"    Projection: Conv2d(3, {model.config.hidden_size}, kernel_size=16, stride=16)")
print(f"    CLS token:  1 learnable vector of dim {model.config.hidden_size}")
print(f"    Position:   {(model.config.image_size // model.config.patch_size)**2 + 1} learnable vectors")
print()
for i, layer in enumerate(model.vit.encoder.layer):
    print(f"  Layer {i}: Self-Attention({model.config.num_attention_heads} heads) → MLP({model.config.hidden_size} → {model.config.intermediate_size} → {model.config.hidden_size})")
print()
print("=" * 60)
print("DECODER (Lightweight)")
print("=" * 60)
print(f"  Embed dim: {model.config.decoder_hidden_size}")
print(f"  Layers:    {model.config.decoder_num_hidden_layers}")
for i, layer in enumerate(model.decoder.decoder_layers):
    print(f"  Layer {i}: Self-Attention({model.config.decoder_num_attention_heads} heads) → MLP")
print(f"  Output:    Linear({model.config.decoder_hidden_size} → {model.config.patch_size**2 * 3})")

### Visualizing Image → Patches

The first step in ViT is **splitting the image into patches**. For a 224×224 image with 16×16 patches:
- We get a **14×14 grid** = **196 patches**
- Each patch captures a small local region of the image
- This is analogous to tokenizing a sentence into words

In [52]:
# ── Visualize how an image becomes patches ──────────────────────────────
IMG_SIZE = 224
PATCH_SIZE = model.config.patch_size  # 16
N_PATCHES_PER_SIDE = IMG_SIZE // PATCH_SIZE  # 14
N_PATCHES = N_PATCHES_PER_SIDE ** 2  # 196

# Pick a chest X-ray
sample_img = to_rgb(train_ds[0][image_col]).resize((IMG_SIZE, IMG_SIZE))
sample_arr = np.array(sample_img)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# (a) Original image
axes[0].imshow(sample_arr)
axes[0].set_title("(a) Original Image\n224 × 224 pixels", fontsize=13)
axes[0].axis("off")

# (b) Image with patch grid overlay
axes[1].imshow(sample_arr)
for i in range(1, N_PATCHES_PER_SIDE):
    axes[1].axhline(y=i * PATCH_SIZE, color="red", linewidth=0.8, alpha=0.7)
    axes[1].axvline(x=i * PATCH_SIZE, color="red", linewidth=0.8, alpha=0.7)
axes[1].set_title(f"(b) Patch Grid\n{N_PATCHES_PER_SIDE}×{N_PATCHES_PER_SIDE} = {N_PATCHES} patches", fontsize=13)
axes[1].axis("off")

# (c) Show individual patches as a sequence (first 28 patches = 2 rows)
n_show_patches = 28
patch_grid = np.ones((2 * (PATCH_SIZE + 2), 14 * (PATCH_SIZE + 2), 3), dtype=np.uint8) * 240
for idx in range(n_show_patches):
    row = idx // 14
    col = idx % 14
    pi, pj = idx // N_PATCHES_PER_SIDE, idx % N_PATCHES_PER_SIDE
    patch = sample_arr[pi*PATCH_SIZE:(pi+1)*PATCH_SIZE, pj*PATCH_SIZE:(pj+1)*PATCH_SIZE]
    y = row * (PATCH_SIZE + 2)
    x = col * (PATCH_SIZE + 2)
    patch_grid[y:y+PATCH_SIZE, x:x+PATCH_SIZE] = patch
axes[2].imshow(patch_grid)
axes[2].set_title(f"(c) First {n_show_patches} patches as a sequence\n(read left→right, top→bottom)", fontsize=13)
axes[2].axis("off")

plt.suptitle("Step 1: Split Image into Patches", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()
print(f"Each patch is {PATCH_SIZE}×{PATCH_SIZE} = {PATCH_SIZE**2} pixels × 3 channels = {PATCH_SIZE**2 * 3}-dim vector")

### From Patches to Embeddings

Each 16×16 patch (768 raw pixel values) is **linearly projected** to a 768-dim embedding vector.
Then a **positional embedding** is added so the model knows where each patch came from.

```
Patch (16×16×3 = 768 values)
     │
     ▼
Linear Projection (768 → 768)    ← Learnable weights
     │
     ▼
Patch Embedding (768-dim)
     │
     + Position Embedding         ← Learnable, unique per position
     │
     ▼
Input to Transformer (768-dim)
```

In [53]:
# ── Visualize the patch embedding process ────────────────────────────────
# Run one image through the model's patch embedding layer
inputs = processor(images=sample_img, return_tensors="pt").to(device)

with torch.no_grad():
    # Get patch embeddings (before Transformer layers)
    patch_embeds = model.vit.embeddings.patch_embeddings(inputs["pixel_values"])
    # patch_embeds shape: (1, 196, 768)

print(f"Input image shape:      {inputs['pixel_values'].shape}")  # (1, 3, 224, 224)
print(f"After patch embedding:  {patch_embeds.shape}")            # (1, 196, 768)
print(f"  → {patch_embeds.shape[1]} patches, each is a {patch_embeds.shape[2]}-dim vector")

# Visualize: show the embedding vectors as a heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# (a) Patch embeddings as heatmap
embed_np = patch_embeds[0].cpu().numpy()
im = axes[0].imshow(embed_np, aspect="auto", cmap="RdBu_r")
axes[0].set_xlabel("Embedding Dimension (768)")
axes[0].set_ylabel("Patch Index (196)")
axes[0].set_title("Patch Embeddings\n(each row = one patch's embedding)")
plt.colorbar(im, ax=axes[0], fraction=0.02)

# (b) Positional embeddings — similarity matrix shows spatial structure
pos_embed = model.vit.embeddings.position_embeddings[0, 1:].detach().cpu()  # skip CLS
# Cosine similarity between positions
pos_norm = pos_embed / pos_embed.norm(dim=-1, keepdim=True)
sim = (pos_norm @ pos_norm.T).numpy()
im2 = axes[1].imshow(sim, cmap="viridis")
axes[1].set_xlabel("Patch Position")
axes[1].set_ylabel("Patch Position")
axes[1].set_title("Positional Embedding Similarity\n(nearby patches have similar embeddings)")
plt.colorbar(im2, ax=axes[1], fraction=0.02)

plt.suptitle("Step 2: Patch Embeddings + Positional Encodings", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## Section 3: How MAE Masking Works

**Masked Autoencoders (He et al., 2021)** train ViT by a simple idea:

1. **Mask ~75%** of the patches (replace with nothing — the encoder never sees them)
2. **Encoder** processes only the **visible 25%** of patches
3. **Decoder** must **reconstruct the missing 75%** from the small visible set
4. **Loss** = mean squared error between reconstructed and original pixels

```
┌──────────────┐      ┌──────────┐      ┌──────────────┐
│  196 patches  │─────▶│  Mask 75% │─────▶│ 49 visible   │
│  (full image) │      │  randomly │      │ patches only │
└──────────────┘      └──────────┘      └──────┬───────┘
                                                │
                                                ▼
                                         ┌──────────────┐
                                         │   Encoder     │  ← Sees only 25%
                                         │   (12 layers) │
                                         └──────┬───────┘
                                                │
                                                ▼
                                         ┌──────────────┐
                                         │   Decoder     │  ← Reconstructs 100%
                                         │   + mask      │
                                         │     tokens    │
                                         └──────┬───────┘
                                                │
                                                ▼
                                         Reconstructed image
```

**Why 75% masking?** With so little visible, the model *cannot* just copy nearby pixels.
It must learn deep semantic understanding — anatomy, symmetry, texture patterns — to
fill in the gaps. This is what makes MAE so powerful for medical images.

In [54]:
# ── Visualize MAE masking on a chest X-ray ──────────────────────────────
MASK_RATIO = 0.75
np.random.seed(SEED)

img = to_rgb(train_ds[0][image_col]).resize((IMG_SIZE, IMG_SIZE))
arr = np.array(img)

# Randomly select patches to mask
ids = np.random.permutation(N_PATCHES)
n_keep = int(N_PATCHES * (1 - MASK_RATIO))
ids_keep = sorted(ids[:n_keep])
ids_mask = sorted(ids[n_keep:])

# Build masked image (gray = masked) and visible-only image (black = masked)
masked_arr = arr.copy()
visible_arr = np.zeros_like(arr)

for idx in range(N_PATCHES):
    i, j = idx // N_PATCHES_PER_SIDE, idx % N_PATCHES_PER_SIDE
    y1, y2 = i * PATCH_SIZE, (i + 1) * PATCH_SIZE
    x1, x2 = j * PATCH_SIZE, (j + 1) * PATCH_SIZE
    if idx in ids_mask:
        masked_arr[y1:y2, x1:x2] = 128  # gray
    else:
        visible_arr[y1:y2, x1:x2] = arr[y1:y2, x1:x2]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(arr);            axes[0].set_title("Original (196 patches)")
axes[1].imshow(masked_arr);     axes[1].set_title(f"Masked (gray = {len(ids_mask)} hidden)")
axes[2].imshow(visible_arr);    axes[2].set_title(f"Encoder input ({n_keep} visible patches)")
for ax in axes:
    ax.axis("off")
plt.suptitle(f"MAE Masking: {n_keep} visible / {N_PATCHES} total ({100*(1-MASK_RATIO):.0f}% visible)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()
print("→ The encoder only sees the colored patches. The decoder must reconstruct everything else.")

---
## Section 4: Reconstructions Before Fine-Tuning

The model was pretrained on **ImageNet** (natural images like cats, cars, landscapes).
Chest X-rays look very different — let's see how well it reconstructs CXR *before*
we fine-tune it on medical data.

In [55]:
def mae_reconstruct(model, processor, images, device):
    'Run MAE forward pass and return reconstructed images as numpy arrays.'
    if not isinstance(images, list):
        images = [images]
    inputs = processor(images=images, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model(**inputs)

    # Reshape logits (B, 196, patch_size^2 * 3) → (B, 224, 224, 3)
    B, N, D = out.logits.shape
    p = model.config.patch_size
    h = w = model.config.image_size // p
    patches = out.logits.view(B, h, w, p, p, 3)
    recon = patches.permute(0, 1, 3, 2, 4, 5).reshape(B, h * p, w * p, 3)

    # Denormalize from ImageNet stats
    mean = torch.tensor([0.485, 0.456, 0.406], device=recon.device)
    std  = torch.tensor([0.229, 0.224, 0.225], device=recon.device)
    recon = recon * std + mean
    recon = recon.clamp(0, 1).cpu().numpy()
    return recon

# Reconstruct validation images BEFORE fine-tuning
n_show = min(4, len(val_ds))
val_imgs = [to_rgb(val_ds[i][image_col]) for i in range(n_show)]
recon_before = mae_reconstruct(model, processor, val_imgs, device)

fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
for i in range(n_show):
    axes[0, i].imshow(val_imgs[i]); axes[0, i].set_title("Original"); axes[0, i].axis("off")
    axes[1, i].imshow(recon_before[i]); axes[1, i].set_title("Reconstructed"); axes[1, i].axis("off")
plt.suptitle("MAE Reconstructions — BEFORE Fine-Tuning\n(Pretrained on ImageNet only)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Helper: extract CLS→patch attention from ViT-MAE encoder (all patches, no masking)
def get_vit_attention_maps(mae_model, proc, images, dev):
    """Bypass MAE masking to get attention over all 196 patches."""
    if not isinstance(images, list):
        images = [images]
    pixel_values = proc(images=images, return_tensors="pt")["pixel_values"].to(dev)

    vit = mae_model.vit
    with torch.no_grad():
        # Patch embeddings + position embeddings (skip CLS position at index 0)
        patch_emb = vit.embeddings.patch_embeddings(pixel_values)
        patch_emb = patch_emb + vit.embeddings.position_embeddings[:, 1:, :]
        # Prepend CLS token with its position embedding
        cls_tokens = vit.embeddings.cls_token + vit.embeddings.position_embeddings[:, :1, :]
        cls_tokens = cls_tokens.expand(patch_emb.shape[0], -1, -1)
        embeddings = torch.cat([cls_tokens, patch_emb], dim=1)

        # Run through encoder layers with attention output
        encoder_out = vit.encoder(embeddings, output_attentions=True)
        # Last layer attention: CLS token → all patches, averaged over heads
        last_attn = encoder_out.attentions[-1]          # (B, heads, seq, seq)
        cls_attn = last_attn[:, :, 0, 1:].mean(dim=1)  # (B, n_patches)

    return cls_attn.cpu().numpy()

# Capture attention maps BEFORE fine-tuning (pretrained on ImageNet only)
model.eval()
attn_before = get_vit_attention_maps(model, processor, val_imgs, device)
print(f"Pretrained attention maps captured: {attn_before.shape}  (images × patches)")

---
## Section 5: Fine-Tune MAE on Chest X-Rays

We use HuggingFace **Trainer** for minimal code. The MAE model's forward pass already
returns a reconstruction `loss` — Trainer uses that automatically.

- **Epochs:** 5 (fast on ~500 images)
- **Batch size:** 16
- **Learning rate:** 1e-4
- **No labels needed** — the loss is purely pixel reconstruction

In [56]:
# Prepare dataset: apply processor to convert images to pixel_values tensors
def process_example(examples):
    # set_transform passes batched examples (dict of lists)
    imgs = examples[image_col]
    if not isinstance(imgs, list):
        imgs = [imgs]
    imgs = [to_rgb(img) for img in imgs]
    inputs = processor(images=imgs, return_tensors="pt")
    return {"pixel_values": inputs["pixel_values"]}

train_ds.set_transform(process_example)
val_ds.set_transform(process_example)

# Verify
ex = train_ds[0]
print(f"pixel_values shape: {ex['pixel_values'].shape}")  # (1, 3, 224, 224)

In [ ]:
if not USE_PRECOMPUTED:
    training_args = TrainingArguments(
        output_dir="./vit_mae_cxr",
        num_train_epochs=5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        learning_rate=1e-4,
        warmup_ratio=0.1,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="no",
        report_to="none",
        fp16=torch.cuda.is_available(),
        remove_unused_columns=False,
        dataloader_num_workers=0,
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
    )
    trainer.train()
    trainer.save_model("./vit_mae_cxr")
    print("Model saved to ./vit_mae_cxr")
else:
    model = ViTMAEForPreTraining.from_pretrained("./vit_mae_cxr").to(device)
    print("Loaded fine-tuned model from ./vit_mae_cxr")

In [ ]:
# Plot training and test loss
if not USE_PRECOMPUTED and trainer.state.log_history:
    train_steps = [e["step"] for e in trainer.state.log_history if "loss" in e]
    train_loss_vals = [e["loss"] for e in trainer.state.log_history if "loss" in e]
    eval_steps = [e["step"] for e in trainer.state.log_history if "eval_loss" in e]
    eval_loss_vals = [e["eval_loss"] for e in trainer.state.log_history if "eval_loss" in e]

    plt.figure(figsize=(8, 4))
    plt.plot(train_steps, train_loss_vals, color="steelblue", linewidth=2, label="Train loss")
    if eval_steps:
        plt.plot(eval_steps, eval_loss_vals, color="coral", linewidth=2, marker="o",
                 markersize=5, label="Test loss")
    plt.xlabel("Step"); plt.ylabel("Loss")
    plt.title("MAE Fine-Tuning: Train and Test Loss on Chest X-Rays")
    plt.legend()
    plt.tight_layout(); plt.show()

---
## Section 6: Reconstructions After Fine-Tuning

After training on chest X-rays, the model should better capture CXR-specific structures:
rib patterns, lung fields, cardiac silhouette, mediastinum.

In [59]:
model.eval()
recon_after = mae_reconstruct(model, processor, val_imgs, device)

fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
for i in range(n_show):
    axes[0, i].imshow(val_imgs[i]); axes[0, i].set_title("Original"); axes[0, i].axis("off")
    axes[1, i].imshow(recon_after[i]); axes[1, i].set_title("Reconstructed"); axes[1, i].axis("off")
plt.suptitle("MAE Reconstructions — AFTER Fine-Tuning on CXR",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## Section 7: Before vs After — The Effect of Domain Adaptation

Side-by-side comparison showing how fine-tuning adapts the model from natural images
to the medical domain.

In [60]:
fig, axes = plt.subplots(3, n_show, figsize=(4 * n_show, 10))
for i in range(n_show):
    axes[0, i].imshow(val_imgs[i]);      axes[0, i].set_title("Original")
    axes[1, i].imshow(recon_before[i]);   axes[1, i].set_title("Before (ImageNet)")
    axes[2, i].imshow(recon_after[i]);    axes[2, i].set_title("After (CXR fine-tuned)")
    for row in range(3):
        axes[row, i].axis("off")
axes[0, 0].set_ylabel("Original", fontsize=12, fontweight="bold")
axes[1, 0].set_ylabel("Before", fontsize=12, fontweight="bold")
axes[2, 0].set_ylabel("After", fontsize=12, fontweight="bold")
plt.suptitle("Domain Adaptation: Before vs After Fine-Tuning on Chest X-Rays",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("→ Fine-tuning on CXR improves reconstruction quality.")
print("  The encoder now produces better representations of medical image features.")
print("  These representations can be used for downstream tasks (classification, segmentation).")

### Attention Maps: Before vs After Fine-Tuning

Beyond reconstruction quality, we can look at **where the model pays attention**. By extracting the CLS token's attention to each patch (last encoder layer, averaged over all heads), we visualize what parts of the image the model considers most important.

- **Before fine-tuning** (ImageNet pretrained): attention may spread across the image or focus on generic patterns
- **After fine-tuning** (CXR adapted): attention should shift toward anatomically relevant regions (lungs, heart, mediastinum)

In [ ]:
# Compute attention maps AFTER fine-tuning
model.eval()
attn_after = get_vit_attention_maps(model, processor, val_imgs, device)
print(f"Fine-tuned attention maps captured: {attn_after.shape}")

grid_size = model.config.image_size // model.config.patch_size  # 14
display_size = 224

fig, axes = plt.subplots(5, n_show, figsize=(4 * n_show, 17))

for col in range(n_show):
    img = val_imgs[col]
    img_resized = np.array(img.resize((display_size, display_size)))

    attn_map_pre = attn_before[col][:grid_size**2].reshape(grid_size, grid_size)
    attn_map_post = attn_after[col][:grid_size**2].reshape(grid_size, grid_size)

    # Row 0: Original image
    axes[0, col].imshow(img)
    axes[0, col].axis('off')

    # Row 1: Before — attention heatmap
    axes[1, col].imshow(attn_map_pre, cmap='inferno')
    axes[1, col].axis('off')

    # Row 2: Before — overlay on image
    attn_uint8 = (attn_map_pre / attn_map_pre.max() * 255).astype(np.uint8)
    attn_resized = np.array(
        Image.fromarray(attn_uint8).resize((display_size, display_size), Image.BILINEAR)
    )
    axes[2, col].imshow(img_resized)
    axes[2, col].imshow(attn_resized, cmap='jet', alpha=0.4)
    axes[2, col].axis('off')

    # Row 3: After — attention heatmap
    axes[3, col].imshow(attn_map_post, cmap='inferno')
    axes[3, col].axis('off')

    # Row 4: After — overlay on image
    attn_uint8 = (attn_map_post / attn_map_post.max() * 255).astype(np.uint8)
    attn_resized = np.array(
        Image.fromarray(attn_uint8).resize((display_size, display_size), Image.BILINEAR)
    )
    axes[4, col].imshow(img_resized)
    axes[4, col].imshow(attn_resized, cmap='jet', alpha=0.4)
    axes[4, col].axis('off')

# Row labels
for row, label in enumerate(["Original", "Before\nAttention", "Before\nOverlay",
                              "After\nAttention", "After\nOverlay"]):
    axes[row, 0].set_ylabel(label, fontsize=11, fontweight='bold')

plt.suptitle("ViT Attention Maps: Before vs After CXR Fine-Tuning\n(Last Layer, Averaged Over All Heads — CLS Token → Patches)",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
---

# Part II: Multimodal Foundation Model (Janus-Pro-1B)

In Part I, we worked with a **vision-only** model (ViT-MAE) that understands images
by reconstructing masked patches.

Now we move to a **multimodal** model that can:
- **Understand** images (describe what it sees, answer questions)
- **Generate** images (create new images from text descriptions)

This is a fundamentally different capability — the model bridges vision and language.

---
## Section 8: Introduction to Multimodal Models

### The Multimodal Challenge

Most AI models are **unimodal** — they work with only one type of data:
- GPT/LLaMA → text only
- ViT/ResNet → images only

**Multimodal models** process multiple modalities (text + images) in a unified architecture.

### Janus-Pro Architecture

**Janus-Pro** (DeepSeek, 2024) uses a clever **decoupled design**:

```
┌─────────────────────────────────────────────────────────┐
│                    Janus-Pro-1B                          │
│                                                         │
│  UNDERSTANDING path:        GENERATION path:            │
│  Image → SigLIP encoder     Text → LLM → Image tokens  │
│       → LLM → Text               → VQ decoder → Image  │
│                                                         │
│  ┌──────────────┐          ┌──────────────┐             │
│  │ SigLIP Vision│          │ VQ Tokenizer │             │
│  │ Encoder      │          │ (gen_vision)  │             │
│  └──────┬───────┘          └──────┬───────┘             │
│         │                         │                     │
│         ▼                         ▼                     │
│  ┌────────────────────────────────────┐                 │
│  │     Shared LLM (1B parameters)     │                 │
│  │     (DeepSeek LLM backbone)        │                 │
│  └────────────────────────────────────┘                 │
│         │                         │                     │
│         ▼                         ▼                     │
│    Text output              Image tokens                │
│                             → VQ Decoder                │
│                             → Generated image           │
└─────────────────────────────────────────────────────────┘
```

**Key idea:** Understanding and generation use **different visual encoders** but share
the same language model backbone. This lets each pathway optimize for its specific task.

---
## Section 9: Load Janus-Pro-1B & Visualize Mixed-Modal Input

Let's load the model and see how it processes images and text together.

In [64]:
# ── Free Part I model to reclaim GPU memory ──────────────────────────────
# The MAE model is no longer needed; releasing it frees ~400 MB on the GPU.
import gc
for _n in ["model", "trainer"]:
    if _n in globals(): del globals()[_n]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Freed MAE model memory.")

# ── Load Janus-Pro-1B ────────────────────────────────────────────────────
from janus.models import MultiModalityCausalLM, VLChatProcessor
from janus.utils.io import load_pil_images

# Patch: transformers>=4.47 uses mark_tied_weights_as_initialized and
# all_tied_weights_keys in ways Janus-Pro doesn't support.
# Override unconditionally so re-running the cell also clears stale values.
MultiModalityCausalLM.mark_tied_weights_as_initialized = lambda self, *a, **kw: None
MultiModalityCausalLM._tied_weights_keys = []
MultiModalityCausalLM.all_tied_weights_keys = {}  # must be dict, not set

JANUS_MODEL_ID = "deepseek-ai/Janus-Pro-1B"

vl_processor = VLChatProcessor.from_pretrained(JANUS_MODEL_ID)
tokenizer = vl_processor.tokenizer

# low_cpu_mem_usage=False avoids meta-tensor init that conflicts with
# Janus's CLIP encoder calling .item() during __init__.
janus_model = MultiModalityCausalLM.from_pretrained(
    JANUS_MODEL_ID, trust_remote_code=True, low_cpu_mem_usage=False
)
janus_model = janus_model.to(torch.bfloat16).to(device).eval()

print(f"Loaded {JANUS_MODEL_ID}")
n_janus_params = sum(p.numel() for p in janus_model.parameters())
print(f"Total parameters: {n_janus_params:,}")

### How Mixed-Modal Input Works

When Janus-Pro receives an image + text prompt, it:
1. **Encodes the image** with SigLIP → sequence of visual tokens
2. **Tokenizes the text** normally
3. **Interleaves** them into a single token sequence for the LLM

Let's visualize this mixed-modal token sequence:

In [68]:
# Visualize the mixed-modal token sequence
# Save a CXR image temporarily for the Janus processor
import tempfile, os

# Reset transform from Part I so we can access raw images again
train_ds.reset_format()
val_ds.reset_format()

sample_cxr = to_rgb(val_ds[0][image_col])
tmp_path = os.path.join(tempfile.gettempdir(), "sample_cxr.png")
sample_cxr.save(tmp_path)

question = "What do you see in this chest X-ray?"

# Build conversation in Janus format
conversation = [
    {
        "role": "<|User|>",
        "content": f"<image_placeholder>\n{question}",
        "images": [tmp_path],
    },
    {"role": "<|Assistant|>", "content": ""},
]

# Process to see the token structure
pil_images = load_pil_images(conversation)
prepare_inputs = vl_processor(
    conversations=conversation, images=pil_images, force_batchify=True
).to(device, dtype=torch.bfloat16)

input_ids = prepare_inputs.input_ids[0].cpu().tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)

# Count token types
# Image tokens are typically represented as special tokens
n_total = len(tokens)
n_special = sum(1 for t in tokens if t.startswith("<") or t.startswith("▁<"))
print(f"Total tokens in sequence: {n_total}")
print(f"Text prompt: \"{question}\"")
print(f"\nFirst 10 tokens: {tokens[:10]}")
print(f"Last 10 tokens:  {tokens[-10:]}")

# Visualize token sequence composition
fig, ax = plt.subplots(figsize=(14, 2))
colors = []
for t in tokens:
    if "image" in t.lower() or t in ["<image_placeholder>"]:
        colors.append("steelblue")  # image tokens
    elif t.startswith("<") or t.startswith("▁<"):
        colors.append("gray")  # special tokens
    else:
        colors.append("coral")  # text tokens

ax.barh(0, len(tokens), color="lightgray", height=0.5)
x = 0
for c in colors:
    ax.barh(0, 1, left=x, color=c, height=0.5, edgecolor="none")
    x += 1
ax.set_xlim(0, len(tokens))
ax.set_yticks([])
ax.set_xlabel("Token Position")
ax.set_title(f"Mixed-Modal Token Sequence ({n_total} tokens)")

legend_elements = [
    mpatches.Patch(color="steelblue", label="Image tokens"),
    mpatches.Patch(color="coral", label="Text tokens"),
    mpatches.Patch(color="gray", label="Special tokens"),
]
ax.legend(handles=legend_elements, loc="upper right")
plt.tight_layout()
plt.show()

---
## Section 10: Image Captioning

The most basic multimodal task: given an image, generate a text description.
For medical images, this is like automated **radiology reporting**.

In [ ]:
def clean_bpe_artifacts(text):
    """Clean byte-level BPE artifacts from decoded text."""
    return text.replace('\u0120', ' ').replace('\u010a', '\n').strip()

@torch.inference_mode()
def janus_understand(model, processor, tokenizer, image, prompt, max_tokens=256):
    'Use Janus-Pro to understand an image and generate text.'
    # Save image to temp file
    tmp = os.path.join(tempfile.gettempdir(), "janus_input.png")
    if isinstance(image, Image.Image):
        image.save(tmp)
    else:
        Image.open(image).save(tmp)

    conversation = [
        {"role": "<|User|>", "content": f"<image_placeholder>\n{prompt}", "images": [tmp]},
        {"role": "<|Assistant|>", "content": ""},
    ]

    pil_images = load_pil_images(conversation)
    inputs = processor(conversations=conversation, images=pil_images, force_batchify=True)
    inputs = inputs.to(model.device, dtype=torch.bfloat16)

    inputs_embeds = model.prepare_inputs_embeds(**inputs)
    outputs = model.language_model.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=inputs.attention_mask,
        pad_token_id=tokenizer.eos_token_id,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        max_new_tokens=max_tokens,
        do_sample=False,
        use_cache=True,
    )
    raw = tokenizer.decode(outputs[0].cpu().tolist(), skip_special_tokens=True)
    return clean_bpe_artifacts(raw)

# Caption several CXR images
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i in range(3):
    img = to_rgb(val_ds[i][image_col])
    caption = janus_understand(janus_model, vl_processor, tokenizer, img,
                               "Describe this chest X-ray image in detail.")
    axes[i].imshow(img, cmap="gray")
    axes[i].set_title(f"Image {i+1}", fontsize=12)
    axes[i].axis("off")
    # Print caption below
    print(f"--- Image {i+1} ---")
    print(f"Caption: {caption}\n")

plt.suptitle("Janus-Pro Image Captioning on Chest X-Rays", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Section 11: Visual Question Answering (VQA)

Beyond captioning, we can ask **specific clinical questions** about the image.
This is the basis for AI-assisted diagnosis tools.

In [70]:
# VQA: ask clinical questions about a CXR
cxr_img = to_rgb(val_ds[0][image_col])

questions = [
    "Is there any abnormality visible in this chest X-ray?",
    "Describe the cardiac silhouette in this image.",
    "Are the lung fields clear?",
]

plt.figure(figsize=(5, 5))
plt.imshow(cxr_img, cmap="gray")
plt.title("Chest X-Ray for VQA")
plt.axis("off")
plt.show()

print("Visual Question Answering Results:")
print("=" * 60)
for q in questions:
    answer = janus_understand(janus_model, vl_processor, tokenizer, cxr_img, q)
    print(f"\nQ: {q}")
    print(f"A: {answer}")

---
## Section 12: Text-to-Image Generation

Janus-Pro can also **generate images from text** using its generation pathway:
1. Text prompt → LLM generates a sequence of **image tokens**
2. Image tokens → **VQ decoder** reconstructs pixel values

This uses **classifier-free guidance (CFG)** — running the model with and without
the text condition, then amplifying the difference to get sharper images.

In [72]:
@torch.inference_mode()
def janus_generate(model, processor, prompt, cfg_weight=5.0, temperature=1.0,
                   image_token_num=576, img_size=384, patch_size=16):
    'Generate an image from a text prompt using Janus-Pro.'
    conversation = [
        {"role": "<|User|>", "content": prompt},
        {"role": "<|Assistant|>", "content": ""},
    ]

    sft_format = processor.apply_sft_template_for_multi_turn_prompts(
        conversations=conversation,
        sft_format=processor.sft_format,
        system_prompt="",
    )
    prompt_text = sft_format + processor.image_start_tag

    input_ids = processor.tokenizer.encode(prompt_text)
    input_ids = torch.LongTensor([input_ids])

    # Duplicate for classifier-free guidance: [conditional, unconditional]
    tokens = torch.zeros((2, len(input_ids[0]) + image_token_num), dtype=torch.long)
    tokens[0, :len(input_ids[0])] = input_ids[0]  # conditional
    tokens[1, :len(input_ids[0])] = input_ids[0]  # unconditional (will be ignored)
    tokens = tokens.to(model.device)

    inputs_embeds = model.language_model.get_input_embeddings()(tokens)

    # Create attention mask
    attention_mask = torch.ones_like(tokens, dtype=torch.long)

    generated_tokens = torch.zeros((1, image_token_num), dtype=torch.int).to(model.device)
    pkv = None

    for i in range(image_token_num):
        if i == 0:
            out = model.language_model.model(
                inputs_embeds=inputs_embeds[:, :len(input_ids[0])],
                use_cache=True,
            )
        else:
            out = model.language_model.model(
                inputs_embeds=inputs_embeds,
                past_key_values=pkv,
                use_cache=True,
            )
        pkv = out.past_key_values
        hidden = out.last_hidden_state[:, -1, :]

        logits = model.gen_head(hidden)
        logit_cond = logits[0:1]
        logit_uncond = logits[1:2]
        logits = logit_uncond + cfg_weight * (logit_cond - logit_uncond)

        probs = torch.softmax(logits / temperature, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        generated_tokens[:, i] = next_token.squeeze(-1)

        # Embed the generated token for the next step
        # next_token is [1,1]; squeeze to [1] so embedding returns [2, hidden]
        # then unsqueeze(1) gives [2, 1, hidden] — correct 3D for inputs_embeds
        next_token_id = next_token.squeeze(-1)
        img_embeds = model.prepare_gen_img_embeds(
            torch.cat([next_token_id, next_token_id], dim=0)
        )
        inputs_embeds = img_embeds.unsqueeze(1)

    # Decode image tokens to pixels
    dec = model.gen_vision_model.decode_code(
        generated_tokens,
        shape=[1, 8, img_size // patch_size, img_size // patch_size],
    )
    dec = dec.to(torch.float32).cpu().numpy().transpose(0, 2, 3, 1)
    dec = np.clip((dec + 1) / 2 * 255, 0, 255).astype(np.uint8)
    return Image.fromarray(dec[0])

# Generate chest X-ray images from text prompts
prompts = [
    "A frontal chest X-ray showing normal lung fields",
    "A chest radiograph with clear costophrenic angles",
    "A PA chest X-ray of a healthy patient",
]

fig, axes = plt.subplots(1, len(prompts), figsize=(5 * len(prompts), 5))
for i, prompt in enumerate(prompts):
    print(f"Generating image {i+1}/{len(prompts)}...")
    gen_img = janus_generate(janus_model, vl_processor, prompt)
    axes[i].imshow(gen_img)
    axes[i].set_title(f"Prompt: {prompt[:40]}...", fontsize=10)
    axes[i].axis("off")

plt.suptitle("Janus-Pro Text-to-Image Generation", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Section 13: Interleaved Generation

A powerful capability of multimodal models is producing **documents that alternate
between text and images** — like a mini radiology report with inline generated views.

We demonstrate this by:
1. **Generating a text description** for a clinical scenario
2. **Generating an image** from that description
3. Combining them into an interleaved report

In [ ]:
# Interleaved generation: text -> image -> text -> image
scenarios = [
    {
        "title": "Normal Chest X-Ray",
        "text_prompt": "Write a brief radiology report for a normal frontal chest X-ray with clear lung fields and normal cardiac silhouette.",
        "image_prompt": "A normal frontal chest X-ray with clear lung fields and normal cardiac silhouette",
    },
    {
        "title": "Chest X-Ray — Follow-up",
        "text_prompt": "Write a brief comparison report for a follow-up chest X-ray showing stable findings.",
        "image_prompt": "A frontal chest X-ray showing stable bilateral lung fields",
    },
]

for scenario in scenarios:
    print("=" * 60)
    print(f"  {scenario['title']}")
    print("=" * 60)

    # Step 1: Generate report text
    # (Using a simple text generation via the model's LLM)
    conversation = [
        {"role": "<|User|>", "content": scenario["text_prompt"]},
        {"role": "<|Assistant|>", "content": ""},
    ]
    sft_format = vl_processor.apply_sft_template_for_multi_turn_prompts(
        conversations=conversation,
        sft_format=vl_processor.sft_format,
        system_prompt="",
    )
    input_ids = tokenizer.encode(sft_format, return_tensors="pt").to(device)
    with torch.inference_mode():
        text_out = janus_model.language_model.generate(
            input_ids=input_ids,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    raw_report = tokenizer.decode(text_out[0][input_ids.shape[1]:], skip_special_tokens=True)
    report_text = clean_bpe_artifacts(raw_report)
    print(f"\nGenerated Report:\n{report_text}\n")

    # Step 2: Generate corresponding image
    print("Generating corresponding X-ray image...")
    gen_img = janus_generate(janus_model, vl_processor, scenario["image_prompt"])

    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.imshow(gen_img)
    ax.set_title(f"Generated: {scenario['title']}")
    ax.axis("off")
    plt.tight_layout()
    plt.show()
    print()

---
## Section 14: Quick Fine-Tuning with LoRA

We can fine-tune Janus-Pro on our CXR dataset to improve its captioning ability.
Using **LoRA (Low-Rank Adaptation)**, we only train a tiny fraction of parameters —
making fine-tuning fast and memory-efficient.

**LoRA** works by adding small trainable matrices to the attention layers:
```
Original weight W (frozen)  +  ΔW = A × B (trainable, low-rank)
```
Instead of updating all 1B parameters, we train ~1M parameters.

In [74]:
from peft import LoraConfig, get_peft_model, TaskType

# Configure LoRA — target the attention layers of the LLM
lora_config = LoraConfig(
    r=8,                       # Rank of the low-rank matrices
    lora_alpha=16,             # Scaling factor
    target_modules=["q_proj", "v_proj"],  # Apply to attention Q and V projections
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA to the language model component
janus_model.language_model = get_peft_model(janus_model.language_model, lora_config)
janus_model.language_model.print_trainable_parameters()

In [75]:
# Prepare training data: image + caption pairs
from torch.utils.data import Dataset, DataLoader

class CXRCaptionDataset(Dataset):
    'Simple dataset that returns tokenized image+caption pairs for Janus.'
    def __init__(self, hf_dataset, processor, tokenizer, image_col, text_col, max_len=128):
        self.dataset = hf_dataset
        self.processor = processor
        self.tokenizer = tokenizer
        self.image_col = image_col
        self.text_col = text_col
        self.max_len = max_len

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        example = self.dataset[idx]
        img = to_rgb(example[self.image_col])
        caption = str(example[self.text_col])[:500]  # Truncate long captions

        # Save image temporarily for processor
        tmp = os.path.join(tempfile.gettempdir(), f"cxr_train_{idx}.png")
        img.save(tmp)

        conversation = [
            {"role": "<|User|>", "content": "<image_placeholder>\nDescribe this chest X-ray.", "images": [tmp]},
            {"role": "<|Assistant|>", "content": caption},
        ]

        pil_images = load_pil_images(conversation)
        inputs = self.processor(
            conversations=conversation, images=pil_images, force_batchify=True
        )
        # Get input_ids and create labels (mask image tokens and prompt, keep caption)
        input_ids = inputs.input_ids[0]
        labels = input_ids.clone()

        return {"input_ids": input_ids, "labels": labels, "attention_mask": inputs.attention_mask[0]}

# Create a small training set (use first 50 examples for quick demo)
n_finetune = min(50, len(train_ds))

# Reset transform (we need raw data, not the MAE transform)
train_ds.reset_format()
val_ds.reset_format()

finetune_ds = CXRCaptionDataset(
    train_ds.select(range(n_finetune)),
    vl_processor, tokenizer, image_col, text_col
)
print(f"Fine-tuning dataset size: {len(finetune_ds)}")
print(f"Sample input_ids shape: {finetune_ds[0]['input_ids'].shape}")

In [ ]:
# Fine-tune with a simple training loop
from torch.optim import AdamW

janus_model.language_model.train()
optimizer = AdamW(janus_model.language_model.parameters(), lr=2e-5)

N_EPOCHS = 2
BATCH_SIZE = 1  # Small batch for memory efficiency

dataloader = DataLoader(finetune_ds, batch_size=BATCH_SIZE, shuffle=True)

# Prepare a small validation set for test loss
n_val_finetune = min(20, len(val_ds))
val_finetune_ds = CXRCaptionDataset(
    val_ds.select(range(n_val_finetune)),
    vl_processor, tokenizer, image_col, text_col
)
val_dataloader = DataLoader(val_finetune_ds, batch_size=BATCH_SIZE, shuffle=False)

train_losses = []
test_losses = []

for epoch in range(N_EPOCHS):
    # --- Training ---
    janus_model.language_model.train()
    epoch_loss = 0
    for step, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # Forward pass through LLM with image embeddings
        try:
            inputs_embeds = janus_model.prepare_inputs_embeds(
                input_ids=input_ids,
                pixel_values=None,  # Already embedded in input_ids by processor
                attention_mask=attention_mask,
            )
        except Exception:
            # Fallback: direct LLM forward
            inputs_embeds = janus_model.language_model.get_input_embeddings()(input_ids)

        outputs = janus_model.language_model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()
        if step % 10 == 0:
            train_losses.append(loss.item())
            print(f"  Epoch {epoch+1}, Step {step}: train loss = {loss.item():.4f}")

    print(f"Epoch {epoch+1} avg train loss: {epoch_loss / len(dataloader):.4f}")

    # --- Test loss ---
    janus_model.language_model.eval()
    val_loss_total = 0
    with torch.no_grad():
        for batch in val_dataloader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            try:
                inputs_embeds = janus_model.prepare_inputs_embeds(
                    input_ids=input_ids, pixel_values=None, attention_mask=attention_mask,
                )
            except Exception:
                inputs_embeds = janus_model.language_model.get_input_embeddings()(input_ids)
            out = janus_model.language_model(
                inputs_embeds=inputs_embeds, attention_mask=attention_mask, labels=labels,
            )
            val_loss_total += out.loss.item()
    avg_val_loss = val_loss_total / len(val_dataloader)
    test_losses.append(avg_val_loss)
    print(f"Epoch {epoch+1} test loss: {avg_val_loss:.4f}")

janus_model.language_model.eval()
print("Fine-tuning complete!")

if train_losses:
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(train_losses, color="steelblue", linewidth=2, label="Train loss (per 10 steps)")
    if test_losses:
        # Plot test loss at epoch boundaries
        epoch_checkpoints = [len(train_losses) // N_EPOCHS * (i + 1) - 1 for i in range(N_EPOCHS)]
        epoch_checkpoints = [min(c, len(train_losses) - 1) for c in epoch_checkpoints]
        ax.scatter(epoch_checkpoints, test_losses, color="coral", s=80, zorder=5, label="Test loss (per epoch)")
    ax.set_xlabel("Logged Step"); ax.set_ylabel("Loss")
    ax.set_title("Janus-Pro LoRA Fine-Tuning: Train and Test Loss")
    ax.legend()
    plt.tight_layout(); plt.show()

In [ ]:
# ── Compare captioning BEFORE vs AFTER LoRA fine-tuning ──────────────────
print("Captioning comparison (Before vs After LoRA):")
print("=" * 60)
for i in range(min(3, len(val_ds))):
    img = to_rgb(val_ds[i][image_col])

    # Caption WITHOUT LoRA (disable adapter)
    janus_model.language_model.disable_adapter_layers()
    caption_before = janus_understand(janus_model, vl_processor, tokenizer, img,
                                      "Describe this chest X-ray image in detail.")
    janus_model.language_model.enable_adapter_layers()

    # Caption WITH LoRA
    caption_after = janus_understand(janus_model, vl_processor, tokenizer, img,
                                     "Describe this chest X-ray image in detail.")

    print(f"\n--- Image {i+1} ---")
    print(f"BEFORE LoRA: {caption_before}")
    print(f"AFTER  LoRA: {caption_after}")

# ── Compare image generation BEFORE vs AFTER LoRA fine-tuning ────────────
gen_prompt = "A frontal chest X-ray showing normal lung fields"
print("\n" + "=" * 60)
print(f"Image generation comparison for: \"{gen_prompt}\"")
print("=" * 60)

# Generate WITHOUT LoRA
janus_model.language_model.disable_adapter_layers()
gen_before = janus_generate(janus_model, vl_processor, gen_prompt)
janus_model.language_model.enable_adapter_layers()

# Generate WITH LoRA
gen_after = janus_generate(janus_model, vl_processor, gen_prompt)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(gen_before)
axes[0].set_title("Before LoRA Fine-Tuning", fontsize=12)
axes[0].axis("off")
axes[1].imshow(gen_after)
axes[1].set_title("After LoRA Fine-Tuning", fontsize=12)
axes[1].axis("off")
plt.suptitle(f"Text-to-Image: \"{gen_prompt}\"", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Section 15: Summary

### Part I — ViT-MAE on Chest X-Rays

| Step | What we did |
|------|-------------|
| 1 | Loaded **chest X-ray** images from HuggingFace |
| 2 | Visualized **ViT architecture**: image → patches → embeddings → Transformer |
| 3 | Understood **MAE masking**: hide 75% of patches, reconstruct from 25% |
| 4 | **Fine-tuned** ViT-MAE on CXR domain with HuggingFace Trainer |
| 5 | Compared **before vs after** — domain adaptation improves reconstruction |

### Part II — Multimodal Foundation Model (Janus-Pro-1B)

| Step | What we did |
|------|-------------|
| 6 | Loaded **Janus-Pro-1B** — a model that understands AND generates images |
| 7 | **Image captioning** — automated description of chest X-rays |
| 8 | **Visual QA** — answering clinical questions about images |
| 9 | **Text-to-image generation** — creating CXR images from text prompts |
| 10 | **Interleaved generation** — producing reports with inline images |
| 11 | **LoRA fine-tuning** — efficient adaptation with <1% trainable parameters |

### Key Takeaways

- **ViT** treats images as sequences of patches — enabling Transformer architectures for vision
- **MAE** is a powerful self-supervised method: mask patches, reconstruct, learn features
- **Multimodal models** like Janus-Pro bridge vision and language in a single architecture
- **LoRA** enables efficient fine-tuning of large models on domain-specific data
- These techniques are directly applicable to **medical imaging** workflows

### Next Steps

- Use the ViT encoder for downstream tasks: CXR classification, disease detection
- Explore larger multimodal models (Janus-Pro-7B, LLaVA-Med)
- Build a complete radiology AI pipeline: image → report → quality check